# Deep Research Tool - Salvage & Debug Notebook

このノートブックは、エラーや接続切れで中断した調査データをサルベージするためのツールです。

## 使い方
1. エラーが発生した後、**カーネルを再起動せずに**このノートブックを開く
2. セルを順番に実行してデータをサルベージ
3. 保存されたファイルを確認

## 1. 現在の変数一覧を確認

In [ ]:
# 現在のメモリ上の変数を確認
print("=" * 60)
print("現在の変数一覧")
print("=" * 60)

for name, obj in sorted(globals().items()):
    if not name.startswith('_'):
        obj_type = type(obj).__name__
        try:
            size = len(obj) if hasattr(obj, '__len__') else '-'
        except:
            size = '-'
        print(f"  {name:30s} | {obj_type:20s} | size: {size}")

## 2. Deep Research Tool 関連オブジェクトの検索

In [ ]:
import gc

print("=" * 60)
print("メモリ上のオブジェクト検索")
print("=" * 60)

# 検索対象のクラス名
target_classes = [
    'Researcher',
    'ResearchSession', 
    'EvidenceLocker',
    'Evidence',
    'ExtractedContent',
    'PageContent',
    'SearchResult',
    'ResearchPlan',
    'TableOfContents',
]

found_objects = {}

for obj in gc.get_objects():
    class_name = type(obj).__name__
    if class_name in target_classes:
        if class_name not in found_objects:
            found_objects[class_name] = []
        found_objects[class_name].append(obj)

print("\n検出されたオブジェクト:")
for class_name, objects in found_objects.items():
    print(f"  ✓ {class_name}: {len(objects)} instances")

if not found_objects:
    print("  ✗ Deep Research Tool関連のオブジェクトが見つかりませんでした")
    print("    → カーネルが再起動されている可能性があります")

## 3. 完全サルベージ実行

In [ ]:
import gc
import json
from pathlib import Path
from datetime import datetime

def salvage_research_data(output_dir: str = None):
    """メモリ上のdeep_research_toolデータをサルベージ"""
    
    if output_dir is None:
        output_dir = f"./salvage_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    salvage_dir = Path(output_dir)
    salvage_dir.mkdir(exist_ok=True)
    
    print("=" * 60)
    print(f"サルベージ開始: {salvage_dir}")
    print("=" * 60)
    
    found_anything = False
    results = {}
    
    # 1. ResearchSession
    sessions = [obj for obj in gc.get_objects() 
                if type(obj).__name__ == 'ResearchSession']
    if sessions:
        found_anything = True
        results['sessions'] = sessions
        for i, s in enumerate(sessions):
            try:
                filepath = salvage_dir / f"session_{i}.json"
                with open(filepath, "w", encoding="utf-8") as f:
                    json.dump(s.to_dict(), f, ensure_ascii=False, indent=2)
                print(f"✓ Session saved: {filepath}")
                print(f"  - ID: {s.session_id}")
                print(f"  - Query: {s.query}")
                print(f"  - State: {s.state}")
                print(f"  - Section Contents: {len(s.section_contents)} sections")
            except Exception as e:
                print(f"✗ Session {i} error: {e}")
    
    # 2. EvidenceLocker
    lockers = [obj for obj in gc.get_objects() 
               if type(obj).__name__ == 'EvidenceLocker']
    if lockers:
        found_anything = True
        results['lockers'] = lockers
        for i, locker in enumerate(lockers):
            try:
                evidence_count = len(locker.get_all_evidence())
                locker.export_to_json(salvage_dir / f"evidence_{i}.json")
                locker.export_to_csv(salvage_dir / f"evidence_{i}.csv")
                print(f"✓ EvidenceLocker saved: evidence_{i}.json/csv")
                print(f"  - Evidence count: {evidence_count}")
            except Exception as e:
                print(f"✗ EvidenceLocker {i} error: {e}")
    
    # 3. Researcher (sessionとevidenceを内包)
    researchers = [obj for obj in gc.get_objects() 
                   if type(obj).__name__ == 'Researcher']
    if researchers:
        found_anything = True
        results['researchers'] = researchers
        for i, r in enumerate(researchers):
            try:
                print(f"\n✓ Researcher {i} found")
                if hasattr(r, 'session') and r.session:
                    filepath = salvage_dir / f"researcher_{i}_session.json"
                    with open(filepath, "w", encoding="utf-8") as f:
                        json.dump(r.session.to_dict(), f, ensure_ascii=False, indent=2)
                    print(f"  - Session saved: {filepath}")
                
                if hasattr(r, 'evidence_locker') and r.evidence_locker:
                    r.evidence_locker.export_to_json(salvage_dir / f"researcher_{i}_evidence.json")
                    r.evidence_locker.export_to_csv(salvage_dir / f"researcher_{i}_evidence.csv")
                    print(f"  - Evidence saved")
            except Exception as e:
                print(f"✗ Researcher {i} error: {e}")
    
    # 4. ExtractedContent
    contents = [obj for obj in gc.get_objects() 
                if type(obj).__name__ == 'ExtractedContent']
    if contents:
        found_anything = True
        results['contents'] = contents
        data = []
        for c in contents:
            try:
                data.append({
                    "source_url": getattr(c, 'source_url', ''),
                    "source_title": getattr(c, 'source_title', ''),
                    "processed_content": getattr(c, 'processed_content', ''),
                    "key_points": getattr(c, 'key_points', []),
                    "relevance_score": getattr(c, 'relevance_score', 0),
                })
            except:
                pass
        
        with open(salvage_dir / "extracted_contents.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\n✓ ExtractedContent saved: {len(data)} items")
    
    # 5. PageContent (生のコンテンツ)
    pages = [obj for obj in gc.get_objects() 
             if type(obj).__name__ == 'PageContent']
    if pages:
        found_anything = True
        results['pages'] = pages
        data = []
        for p in pages:
            try:
                data.append({
                    "url": getattr(p, 'url', ''),
                    "title": getattr(p, 'title', ''),
                    "text_content": getattr(p, 'text_content', '')[:10000],
                    "metadata": getattr(p, 'metadata', {}),
                })
            except:
                pass
        
        with open(salvage_dir / "page_contents.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ PageContent saved: {len(data)} items")
    
    # 6. SearchResult
    search_results = [obj for obj in gc.get_objects() 
                      if type(obj).__name__ == 'SearchResult']
    if search_results:
        found_anything = True
        results['search_results'] = search_results
        data = []
        for r in search_results:
            try:
                data.append({
                    "url": getattr(r, 'url', ''),
                    "title": getattr(r, 'title', ''),
                    "snippet": getattr(r, 'snippet', ''),
                })
            except:
                pass
        
        with open(salvage_dir / "search_results.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ SearchResult saved: {len(data)} items")
    
    # 7. Evidence (個別)
    evidences = [obj for obj in gc.get_objects() 
                 if type(obj).__name__ == 'Evidence']
    if evidences:
        found_anything = True
        results['evidences'] = evidences
        data = []
        for e in evidences:
            try:
                data.append(e.to_dict())
            except:
                try:
                    data.append({
                        "url": getattr(e, 'url', ''),
                        "title": getattr(e, 'title', ''),
                        "content_excerpt": getattr(e, 'content_excerpt', ''),
                        "relevance_score": getattr(e, 'relevance_score', 0),
                    })
                except:
                    pass
        
        with open(salvage_dir / "evidences.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ Evidence saved: {len(data)} items")
    
    print("\n" + "=" * 60)
    if found_anything:
        print(f"サルベージ完了: {salvage_dir}")
        print("\n保存されたファイル:")
        for f in sorted(salvage_dir.glob("*")):
            print(f"  - {f.name} ({f.stat().st_size:,} bytes)")
    else:
        print("サルベージ可能なデータが見つかりませんでした")
        print("Jupyterカーネルが再起動されている可能性があります")
    print("=" * 60)
    
    return results if found_anything else None

# サルベージ実行
salvaged = salvage_research_data()

## 4. サルベージ結果の確認

In [ ]:
# サルベージされたセッションの詳細確認
if salvaged and 'sessions' in salvaged:
    session = salvaged['sessions'][0]
    
    print("=" * 60)
    print("セッション詳細")
    print("=" * 60)
    print(f"Session ID: {session.session_id}")
    print(f"Query: {session.query}")
    print(f"State: {session.state}")
    print(f"Started: {session.started_at}")
    print(f"Completed: {session.completed_at}")
    
    print("\n【生成済みセクション】")
    for key, content in session.section_contents.items():
        text = content.get('content', '')
        print(f"  [{key}] {len(text):,} chars")
    
    print("\n【反復記録】")
    for it in session.iterations:
        print(f"  Section {it.section}: {it.content_extracted} contents")
else:
    print("セッションデータがありません")

In [ ]:
# サルベージされた本文の確認
if salvaged and 'sessions' in salvaged:
    session = salvaged['sessions'][0]
    
    print("=" * 60)
    print("生成済み本文")
    print("=" * 60)
    
    for key in sorted(session.section_contents.keys()):
        content = session.section_contents[key]
        text = content.get('content', '')
        
        print(f"\n### {key} ###")
        print(text[:3000])
        if len(text) > 3000:
            print(f"\n... (以下省略、全{len(text):,}文字)")
        print("-" * 40)

## 4.5 報告書本文の抽出・保存

生成された報告書の本文をテキストファイルやWord文書として保存できます。

In [ ]:
# ============================================================
# 報告書本文の抽出・保存
# ============================================================

import json
from pathlib import Path
from datetime import datetime


def extract_report_text(
    source,
    output_path: str = None,
    encoding: str = "utf-8",
    format: str = "markdown",  # "markdown", "txt", "docx"
):
    """
    サルベージしたデータから報告書本文を抽出・保存
    
    Args:
        source: 以下のいずれか
            - salvaged['sessions'][0]: サルベージしたセッションオブジェクト
            - session_data (dict): JSONから読み込んだセッションデータ
            - str: セッションJSONファイルパス
        output_path: 出力ファイルパス（Noneの場合は自動生成）
        encoding: 出力エンコーディング ("utf-8", "utf-8-sig", "cp932", "shift_jis")
        format: 出力形式 ("markdown", "txt", "docx")
    
    Returns:
        str: 出力ファイルパス
    """
    # ソースデータを取得
    if isinstance(source, str):
        # ファイルパスの場合
        with open(source, 'r', encoding='utf-8') as f:
            data = json.load(f)
    elif hasattr(source, 'to_dict'):
        # セッションオブジェクトの場合
        data = source.to_dict()
    elif isinstance(source, dict):
        # 辞書の場合
        data = source
    else:
        raise ValueError(f"Unsupported source type: {type(source)}")
    
    # セクション内容を取得
    section_contents = data.get('section_contents', {})
    
    if not section_contents:
        print("✗ セクション内容が見つかりません")
        return None
    
    # タイトルと概要
    query = data.get('query', 'Unknown Query')
    title = data.get('title', query)
    
    # 本文を組み立て
    lines = []
    
    if format == "markdown":
        lines.append(f"# {title}\n")
        lines.append(f"Research Query: {query}\n")
        lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        lines.append("---\n\n")
    else:
        lines.append(f"{title}\n")
        lines.append("=" * len(title) + "\n\n")
        lines.append(f"Research Query: {query}\n")
        lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    # セクションをソートして追加
    for section_key in sorted(section_contents.keys()):
        section = section_contents[section_key]
        section_title = section.get('title', section_key)
        content = section.get('content', '')
        
        if not content:
            continue
        
        if format == "markdown":
            # セクション番号に応じてヘッダーレベルを決定
            if '.' not in section_key:
                lines.append(f"## {section_key}. {section_title}\n\n")
            elif section_key.count('.') == 1:
                lines.append(f"### {section_key} {section_title}\n\n")
            else:
                lines.append(f"#### {section_key} {section_title}\n\n")
        else:
            lines.append(f"\n{section_key}. {section_title}\n")
            lines.append("-" * 40 + "\n\n")
        
        lines.append(content + "\n\n")
    
    full_text = "".join(lines)
    
    # 出力パスを決定
    if output_path is None:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        ext = "md" if format == "markdown" else format
        output_path = f"./report_{timestamp}.{ext}"
    
    # 保存
    if format == "docx":
        try:
            from docx import Document
            doc = Document()
            doc.add_heading(title, 0)
            
            for section_key in sorted(section_contents.keys()):
                section = section_contents[section_key]
                section_title = section.get('title', section_key)
                content = section.get('content', '')
                
                if not content:
                    continue
                
                level = 1 if '.' not in section_key else (2 if section_key.count('.') == 1 else 3)
                doc.add_heading(f"{section_key}. {section_title}", level)
                doc.add_paragraph(content)
            
            doc.save(output_path)
            print(f"✓ DOCX saved: {output_path}")
        except ImportError:
            print("✗ python-docx がインストールされていません")
            print("  pip install python-docx")
            # フォールバック: テキストで保存
            output_path = output_path.replace('.docx', '.txt')
            with open(output_path, 'w', encoding=encoding) as f:
                f.write(full_text)
            print(f"✓ Text fallback: {output_path}")
    else:
        with open(output_path, 'w', encoding=encoding, errors='replace') as f:
            f.write(full_text)
        print(f"✓ {format.upper()} saved: {output_path} (encoding: {encoding})")
    
    # 統計情報
    total_chars = sum(len(s.get('content', '')) for s in section_contents.values())
    print(f"  - Sections: {len(section_contents)}")
    print(f"  - Total chars: {total_chars:,}")
    
    return output_path


def preview_report_sections(source, max_chars: int = 500):
    """
    報告書セクションのプレビュー表示
    
    Args:
        source: セッションオブジェクト、辞書、またはJSONファイルパス
        max_chars: 各セクションの表示文字数上限
    """
    # ソースデータを取得
    if isinstance(source, str):
        with open(source, 'r', encoding='utf-8') as f:
            data = json.load(f)
    elif hasattr(source, 'to_dict'):
        data = source.to_dict()
    elif isinstance(source, dict):
        data = source
    else:
        raise ValueError(f"Unsupported source type: {type(source)}")
    
    section_contents = data.get('section_contents', {})
    
    if not section_contents:
        print("✗ セクション内容が見つかりません")
        return
    
    print("=" * 60)
    print("報告書セクション プレビュー")
    print("=" * 60)
    
    for section_key in sorted(section_contents.keys()):
        section = section_contents[section_key]
        title = section.get('title', section_key)
        content = section.get('content', '')
        
        print(f"\n【{section_key}】{title}")
        print(f"  文字数: {len(content):,}")
        
        if content:
            preview = content[:max_chars].replace('\n', ' ')
            if len(content) > max_chars:
                preview += "..."
            print(f"  内容: {preview}")
        else:
            print("  内容: (空)")
        print("-" * 40)


print("=== 報告書抽出関数を定義しました ===")
print()
print("使用例:")
print("  # サルベージしたセッションから抽出")
print("  extract_report_text(salvaged['sessions'][0], 'report.md')")
print()
print("  # JSONファイルから抽出")
print("  extract_report_text('salvage_xxx/session_0.json', 'report.md')")
print()
print("  # cp932エンコーディングでテキスト保存")
print("  extract_report_text(session, 'report.txt', encoding='cp932', format='txt')")
print()
print("  # Word文書として保存（python-docxが必要）")
print("  extract_report_text(session, 'report.docx', format='docx')")

## 5. 既存の出力ディレクトリを確認

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def check_output_directory(output_dir: str = "./output"):
    """出力ディレクトリの既存ファイルを確認"""
    
    output_path = Path(output_dir)
    
    if not output_path.exists():
        print(f"Output directory not found: {output_dir}")
        return None
    
    print("=" * 60)
    print(f"出力ディレクトリ確認: {output_dir}")
    print("=" * 60)
    
    # 全ファイル一覧
    print("\n【ファイル一覧】")
    all_files = list(output_path.rglob("*"))
    files = [f for f in all_files if f.is_file()]
    
    if not files:
        print("  ファイルがありません")
        return None
    
    for f in sorted(files, key=lambda x: x.stat().st_mtime, reverse=True):
        mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  {mtime} | {f.relative_to(output_path)} ({f.stat().st_size:,} bytes)")
    
    return files

# 確認実行
existing_files = check_output_directory("./output")

In [ ]:
# 既存のセッションファイルを読み込む
from pathlib import Path
import json

output_dir = Path("./output")
session_files = list(output_dir.glob("session_*.json"))

if session_files:
    latest_session = max(session_files, key=lambda x: x.stat().st_mtime)
    print(f"Loading: {latest_session}")
    
    with open(latest_session, "r", encoding="utf-8") as f:
        session_data = json.load(f)
    
    print(f"\nSession ID: {session_data.get('session_id')}")
    print(f"Query: {session_data.get('query')}")
    print(f"State: {session_data.get('state')}")
    
    section_contents = session_data.get('section_contents', {})
    print(f"\n生成済みセクション: {len(section_contents)}")
    for key, content in section_contents.items():
        print(f"  [{key}] {len(content.get('content', '')):,} chars")
else:
    print("セッションファイルが見つかりません")

## 6. Jupyter 履歴から復元

In [ ]:
# Jupyterの出力履歴を確認
print("=== 出力履歴 (Out) ===")
if '_oh' in dir():
    for key, value in sorted(_oh.items()):
        value_type = type(value).__name__
        print(f"  Out[{key}]: {value_type}")
        
        # deep_research_tool関連なら詳細表示
        if value_type in ['ResearchSession', 'EvidenceLocker', 'dict']:
            print(f"    → 復元可能な可能性あり")
else:
    print("  出力履歴がありません")

print("\n=== 入力履歴 (最新10件) ===")
if '_ih' in dir():
    for i, cmd in enumerate(_ih[-10:]):
        idx = len(_ih) - 10 + i
        preview = cmd[:80].replace('\n', ' ')
        print(f"  In[{idx}]: {preview}...")
else:
    print("  入力履歴がありません")

## 7. 手動復元用ユーティリティ

In [ ]:
# 特定の変数名でオブジェクトを探す
def find_variable(name: str):
    """グローバル変数から特定の名前を探す"""
    if name in globals():
        obj = globals()[name]
        print(f"Found: {name}")
        print(f"  Type: {type(obj).__name__}")
        return obj
    else:
        print(f"Not found: {name}")
        return None

# よくある変数名を検索
common_names = ['researcher', 'session', 'result', 'evidence_locker', 'locker', 'searcher', 'client']
print("=== 一般的な変数名の検索 ===")
for name in common_names:
    find_variable(name)

In [ ]:
# 特定のオブジェクトからデータを抽出
def extract_data_from_object(obj, save_path: str = None):
    """オブジェクトからデータを抽出して保存"""
    import json
    
    obj_type = type(obj).__name__
    print(f"Extracting from: {obj_type}")
    
    data = None
    
    if hasattr(obj, 'to_dict'):
        data = obj.to_dict()
    elif isinstance(obj, dict):
        data = obj
    elif hasattr(obj, '__dict__'):
        data = {k: str(v)[:1000] for k, v in obj.__dict__.items()}
    
    if data and save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
        print(f"Saved to: {save_path}")
    
    return data

# 使用例:
# extract_data_from_object(researcher.session, "manual_salvage.json")

In [ ]:
# ============================================================
# CSV エクスポート（エンコーディング選択可能）
# ============================================================

import csv
import json
from pathlib import Path

def export_to_csv(
    data: list,
    output_path: str,
    encoding: str = "utf-8-sig",  # デフォルトはExcel対応BOM付きUTF-8
    fieldnames: list = None,
):
    """
    データをCSVにエクスポート（エンコーディング選択可能）
    
    Args:
        data: 辞書のリスト
        output_path: 出力ファイルパス
        encoding: エンコーディング
            - "utf-8-sig": BOM付きUTF-8（Excel対応、推奨）
            - "utf-8": UTF-8
            - "cp932": Windows日本語（Shift_JIS互換）
            - "shift_jis": Shift_JIS
        fieldnames: CSVのカラム名（Noneの場合は自動検出）
    """
    if not data:
        print("データが空です")
        return
    
    # フィールド名を自動検出
    if fieldnames is None:
        fieldnames = list(data[0].keys())
    
    with open(output_path, 'w', newline='', encoding=encoding, errors='replace') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for row in data:
            # 値を文字列に変換（リストや辞書は JSON文字列に）
            cleaned_row = {}
            for k, v in row.items():
                if isinstance(v, (list, dict)):
                    cleaned_row[k] = json.dumps(v, ensure_ascii=False)[:1000]
                elif v is None:
                    cleaned_row[k] = ""
                else:
                    cleaned_row[k] = str(v)[:5000]
            writer.writerow(cleaned_row)
    
    print(f"✓ CSV saved: {output_path} (encoding: {encoding})")


def convert_json_to_csv(
    json_path: str,
    csv_path: str = None,
    encoding: str = "cp932",
):
    """
    JSONファイルをCSVに変換（エンコーディング指定可能）
    
    Args:
        json_path: 入力JSONファイルパス
        csv_path: 出力CSVファイルパス（Noneの場合は自動生成）
        encoding: 出力エンコーディング
    """
    json_path = Path(json_path)
    
    if csv_path is None:
        csv_path = json_path.with_suffix(f'.{encoding}.csv')
    
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # dataがリストでない場合（単一オブジェクト）
    if isinstance(data, dict):
        if 'evidence' in data:
            data = data['evidence']
        else:
            data = [data]
    
    export_to_csv(data, str(csv_path), encoding=encoding)
    return csv_path


print("=== CSV エクスポート関数を定義しました ===")

## 8. CSV エクスポート（エンコーディング選択可能）

文字化け対策として、CSVを任意のエンコーディングで保存できます。

| エンコーディング | 説明 | 用途 |
|----------------|------|------|
| `utf-8-sig` | BOM付きUTF-8 | Excel（推奨） |
| `utf-8` | UTF-8 | 汎用 |
| `cp932` | Windows日本語 | 古いWindowsアプリ |
| `shift_jis` | Shift_JIS | レガシーシステム |

In [ ]:
# ============================================================
# サルベージディレクトリ内の全JSONをCSVに一括変換
# ============================================================

def batch_convert_to_csv(
    salvage_dir: str,
    encoding: str = "cp932",
):
    """
    サルベージディレクトリ内の全JSONファイルをCSVに変換
    
    Args:
        salvage_dir: サルベージディレクトリのパス
        encoding: 出力エンコーディング
    """
    salvage_path = Path(salvage_dir)
    
    if not salvage_path.exists():
        print(f"ディレクトリが見つかりません: {salvage_dir}")
        return
    
    json_files = list(salvage_path.glob("*.json"))
    
    if not json_files:
        print(f"JSONファイルが見つかりません: {salvage_dir}")
        return
    
    print(f"=== 一括変換: {salvage_dir} ===")
    print(f"エンコーディング: {encoding}")
    print()
    
    for json_file in json_files:
        try:
            csv_path = convert_json_to_csv(str(json_file), encoding=encoding)
            print(f"  {json_file.name} → {Path(csv_path).name}")
        except Exception as e:
            print(f"  ✗ {json_file.name}: {e}")
    
    print()
    print("変換完了！")


# ============================================================
# 使用例: サルベージディレクトリを指定して一括変換
# ============================================================

# 最新のサルベージディレクトリを自動検出
salvage_dirs = sorted(Path(".").glob("salvage_*"), key=lambda x: x.stat().st_mtime, reverse=True)

if salvage_dirs:
    latest_salvage = salvage_dirs[0]
    print(f"最新のサルベージディレクトリ: {latest_salvage}")
    print()
    print("以下を実行してcp932 CSVに変換できます:")
    print(f'  batch_convert_to_csv("{latest_salvage}", encoding="cp932")')
else:
    print("サルベージディレクトリが見つかりません")
    print("先にセクション3のサルベージを実行してください")

---

## 補足: 文字化け対策まとめ

### エンコーディングの選び方

| 用途 | 推奨エンコーディング | コマンド |
|-----|-------------------|---------|
| Excel で開く | `utf-8-sig` | `encoding="utf-8-sig"` |
| 古いWindowsアプリ | `cp932` | `encoding="cp932"` |
| プログラムで処理 | `utf-8` | `encoding="utf-8"` |

### 使用例

```python
# 1. 単一のJSONファイルをcp932 CSVに変換
convert_json_to_csv("salvage_xxx/evidences.json", encoding="cp932")

# 2. サルベージディレクトリ内の全JSONを一括変換
batch_convert_to_csv("salvage_xxx", encoding="cp932")

# 3. データを直接CSVに保存
export_to_csv(data_list, "output.csv", encoding="cp932")
```

### 今後のエラー対策

```python
from deep_research_tool import run_research

result = run_research(
    query="テーマ",
    verbose=True,  # 進捗表示
    output_dir="./output",  # 明示的に出力先指定
)
```